# CE49X Midterm Exam - Part 2 (Coding Exercise)
## Power Grid Stability Prediction

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Date:** April 8, 2026  
**Duration:** 60 minutes  
**Total Points:** 50 (+3 bonus)

---

**Student Name:**  
**Student ID:**

## Background

Electrical grid stability is a critical infrastructure challenge. As cities grow and energy consumption patterns become more complex, maintaining a stable power grid requires understanding how different factors — such as reaction times of energy producers, power consumption patterns, and price sensitivity of consumers — affect overall grid stability.

In this exercise, you will work with the **UCI Electrical Grid Stability** dataset, which contains 10,000 simulated scenarios of a 4-node star power grid. The grid consists of **one energy producer** (node 1) connected to **three consumers** (nodes 2, 3, 4). Each scenario records the operating parameters of all four nodes and whether the grid remained **stable** or became **unstable**.

Your task is to explore the data, identify which factors most influence grid stability, and build a classifier to predict whether a given configuration will be stable or unstable.

> **Key Insight:** This is an infrastructure safety problem. An unstable grid can lead to blackouts, equipment damage, and cascading failures. The cost of failing to detect instability is far greater than the cost of a false alarm.

## Dataset Description

The dataset is from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/471/) (Dataset #471).

| Feature | Description | Unit |
|---------|-------------|------|
| `tau1` | Reaction time of producer (node 1) | seconds |
| `tau2` | Reaction time of consumer 2 | seconds |
| `tau3` | Reaction time of consumer 3 | seconds |
| `tau4` | Reaction time of consumer 4 | seconds |
| `p1` | Power produced by node 1 | per unit |
| `p2` | Power consumed by node 2 | per unit |
| `p3` | Power consumed by node 3 | per unit |
| `p4` | Power consumed by node 4 | per unit |
| `g1` | Price elasticity coefficient of producer | dimensionless |
| `g2` | Price elasticity coefficient of consumer 2 | dimensionless |
| `g3` | Price elasticity coefficient of consumer 3 | dimensionless |
| `g4` | Price elasticity coefficient of consumer 4 | dimensionless |
| `stab` | Stability measure (continuous) | — |
| **`stabf`** | **Stability label: `stable` or `unstable`** | **— (target)** |

- **Positive `stab`** values indicate instability; negative values indicate stability
- **Power balance:** `p1 + p2 + p3 + p4` should be close to zero (production = consumption)

## Tasks Overview

| # | Task | Points |
|---|------|--------|
| 1 | Data Loading & Exploration | 8 |
| 2 | Feature Engineering | 8 |
| 3 | Grouped Analysis | 10 |
| 4 | Visualization | 12 |
| 5 | Statistical Analysis | 6 |
| 6 | Classification | 6 |
| WQ1 | Written Question 1 | 3 |
| WQ2 | Written Question 2 (Bonus) | 3 |
| **Total** | | **50 (+3 bonus)** |

---
## Your Work Starts Here

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

---
## Task 1: Data Loading & Exploration (8 pts)

1. Load `data/electrical_grid_stability.csv` into a DataFrame
2. Print the shape and data types
3. Display the first 5 rows
4. Check for missing values
5. Print `.describe()` for all numeric columns
6. Print the value counts of `stabf` (the target variable)

In [ ]:
df = pd.read_csv('data/electrical_grid_stability.csv')

In [ ]:
print(f"Shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDescriptive statistics:")
display(df.describe())
print("\nValue counts of stabf:")
print(df['stabf'].value_counts())

---
## Task 2: Feature Engineering (8 pts)

1. Convert `stabf` to a numeric column called `is_unstable`: 1 if `unstable`, 0 if `stable`
2. Create a new column `total_reaction_time` = `tau1 + tau2 + tau3 + tau4`
3. Create a new column `power_imbalance` = `p1 + p2 + p3 + p4`
4. Create a new column `avg_elasticity` = mean of `g1, g2, g3, g4`
5. Print the class balance: count and percentage of stable vs unstable

In [ ]:
df['is_unstable'] = df['stabf'].map({'unstable': 1, 'stable': 0})
df['total_reaction_time'] = df['tau1'] + df['tau2'] + df['tau3'] + df['tau4']
df['power_imbalance'] = df['p1'] + df['p2'] + df['p3'] + df['p4']
df['avg_elasticity'] = df[['g1', 'g2', 'g3', 'g4']].mean(axis=1)

In [ ]:
counts = df['stabf'].value_counts()
pcts = df['stabf'].value_counts(normalize=True) * 100
balance = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
print("Class Balance:")
print(balance)

---
## Task 3: Grouped Analysis (10 pts)

1. Compute the mean of all 12 original features (`tau1`-`tau4`, `p1`-`p4`, `g1`-`g4`) grouped by `stabf`.
2. Compute the correlation of all 12 original features with `stab`. Identify the **3 features** with the highest absolute correlation.
3. Filter to **unstable grids only**: report the range (min, max) and mean of `tau1`.
4. Compare `g1` statistics between stable and unstable grids.

In [ ]:
original_features = ['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2', 'g3', 'g4']

print("Grouped Means:")
display(df.groupby('stabf')[original_features].mean())

print("\nCorrelations with 'stab' (Top 3):")
corrs = df[original_features + ['stab']].corr()['stab'].abs().sort_values(ascending=False)
print(corrs[1:4])

unstable_df = df[df['stabf'] == 'unstable']
print(f"\nUnstable Grid tau1 Range: ({unstable_df['tau1'].min():.4f}, {unstable_df['tau1'].max():.4f})")
print(f"Unstable Grid tau1 Mean: {unstable_df['tau1'].mean():.4f}")

print("\ng1 Statistics by group:")
print(df.groupby('stabf')['g1'].agg(['mean', 'std']))

---
## Task 4: Visualization (12 pts)

Create **three** publication-quality plots.

**(a)** A **boxplot** of `tau1` grouped by `stabf`.
**(b)** A **scatter plot** of `tau1` vs `g1`, colored by `stabf`.
**(c)** A **correlation heatmap** of all 12 original features.

In [ ]:
# (a) Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(x='stabf', y='tau1', data=df)
plt.title('Unstable Grids Exhibit Significantly Higher Producer Reaction Times (tau1)')
plt.xlabel('Stability Label (stabf)')
plt.ylabel('Producer Reaction Time (tau1) [s]')
plt.show()

# (b) Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='tau1', y='g1', hue='stabf', data=df, alpha=0.5, s=15)
plt.title('Producer Reaction Time vs Elasticity Coefficient')
plt.legend(title='Stability')
plt.show()

# (c) Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df[original_features].corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation Heatmap of Grid Parameters')
plt.show()

---
## Task 5: Statistical Analysis (6 pts)

1. Compute z-scores for `tau1`. Identify samples where |z| > 2.
2. Perform a **two-sample t-test** comparing `tau1` between stable and unstable grids.
3. Based on analysis, identify the most predictive feature.

In [ ]:
df['tau1_z'] = stats.zscore(df['tau1'])
outliers = df[df['tau1_z'].abs() > 2]
print(f"Samples with |z| > 2 for tau1: {len(outliers)}")

tau1_stable = df[df['stabf'] == 'stable']['tau1']
tau1_unstable = df[df['stabf'] == 'unstable']['tau1']
t_stat, p_val = stats.ttest_ind(tau1_stable, tau1_unstable)
print(f"\nT-test Result: t-stat={t_stat:.4f}, p-value={p_val:.4e}")

print("\nInterpretation: Since p-value < 0.05, we reject H0. tau1 differs significantly between groups.")
print("Most predictive feature: tau1 (highest absolute correlation with continuous stability measure 'stab').")

---
## Task 6: Classification (6 pts)

1. Define `X` (original features) and `y` (`is_unstable`).
2. Split (80/20, stratify, random_state=42).
3. Scale (fit on train only).
4. Train `LogisticRegression`.
5. Report metrics and confusion matrix.

In [ ]:
X = df[original_features]
y = df['is_unstable']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Unstable): {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Unstable): {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score (Unstable): {f1_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

---
## Written Questions

### Written Question 1 (3 pts)

Look at your confusion matrix from Task 6. In the context of **power grid management**, which error is more dangerous:

- **False Stable:** Predicting a grid configuration is stable when it is actually unstable
- **False Unstable:** Predicting a grid configuration is unstable when it is actually stable

**Answer:** **False Stable** is far more dangerous. If an operator assumes the grid is stable when it is actually nearing a point of failure, they will take no corrective action, potentially leading to catastrophic blackouts and equipment damage. A False Unstable prediction only leads to unnecessary preventative measures and costs. Therefore, **recall** for the "unstable" class must be prioritized.

### Written Question 2 — BONUS (3 pts)

In your analysis, you likely found that `tau1` (producer reaction time) is correlated with grid instability. Does this prove that slow producer reaction times **cause** instability?

**Answer:** No, correlation does not imply causation. While they are mathematically related, a **confounding variable** like **Grid Load** could explain the relationship: as demand increases, the system naturally becomes less stable due to physical constraints, while the producer's control systems simultaneously become slower to respond due to the high stress on the machinery.

---

### End of Part 2

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr